# McCall Model

Pablo Winant

### Job-Search Model

- When unemployed in date, a job-seeker
  - consumes unemployment benefit $c_t = \underline{c}$
  - receives in every date $t$ a job offer $w_t$
    - $w_t$ is i.i.d.,
    - takes values $w_1, w_2, w_3$ with probabilities $p_1, p_2, p_3$
  - if job-seeker accepts, becomes employed at rate $w_t$ in the next
    period
  - else he stays unemployed
- When employed at rate $w$
  - worker consumes salary $c_t = w$
  - with small probability $\lambda>0$ looses his job:
    - starts next period unemployed
  - otherwise stays employed at same rate
- Objective: $\max E_0 \left\{ \sum \beta^t \log(c_t) \right\}$

1.  **What are the states, the controls, the reward of this problem ?
    Write down the Bellman equation.**

The state variables are the employment status ("employed"/"unemployed") and the 3 wage rate / job offer.
In total it yields 3x2=6 states.

The only control is the decision to accept/reject when unemployed (2 choices in the 3 unemployed states).

The reward is utility of consumption, that is $U(\underline{c})$ if unemployed, and $U(w)$ if employed.


Bellman equation: as in the slides.

1.  **Define a named tuple for the model.**

In [3]:
model = (; β=0.96, λ=0.05, w=[0.9, 1.0, 1.1], p=[1,1,1]/3, cbar=0.8  )

(β = 0.96, λ = 0.05, w = [0.9, 1.0, 1.1], p = [0.3333333333333333, 0.3333333333333333, 0.3333333333333333], cbar = 0.8)

1.  **Define a function
    `value_update(V_U::Vector{Float64}, V_E::Vector{Float64}, x::Vector{Bool}, p::Parameters)::Tuple{Vector, Vector}`,
    which takes in value functions tomorrow and a policy vector and
    return updated values for today.**

In [89]:
function value_update(V_U, V_E, x, model)

    #extract parameters form model
    (;β, λ, w, p, cbar) = model

    # initialize return variables
    n_V_E = V_E*0
    n_V_U = V_U*0

    n = length(V_U) # here 3

    # continuation value of being unemployed tomorrow
    # EVU = sum(V_U'*p)
    EVU = sum( (V_U[k]*p[k] for k=1:n) )

    # update employed values
    for k=1:n
        n_V_E[k] = log(w[k]) + (1-λ)*β*V_E[k] + λ*β*EVU
    end

    # update uneployed values
    for k=1:n
        n_V_U[k] = log(cbar)
        if x[k] # if accept
            n_V_U[k] += β*V_E[k]
        else # if reject
            n_V_U[k] += β*EVU
        end
    end

    return n_V_U, n_V_E
end

value_update (generic function with 1 method)

In [16]:
V_U_0 = [0.1, 0.1, 0.2]
V_E_0 = [0.1, 0.1, 0.2]
x_0 = [true, false, false]

value_update(V_U_0, V_E_0, x_0, model)

([-0.1271435513142097, -0.0951435513142097, -0.0951435513142097], [-0.007760515657826278, 0.0976, 0.28411017980432496])

1.  **Define a function
    `policy_eval(x::Vector{Bool}, p::Parameter)::Tuple{Vector, Vector}`
    which takes in a policy vector and returns the value(s) of following
    this policies forever. You can add relevant arguments to the
    function.**

In [ ]:
v = [3,2,3]
using LinearAlgebra: norm
norm(v)  #euclidean distance

4.69041575982343

In [ ]:
distance(u, v) = sqrt(
    norm(u[1]-v[1])^2 + norm(u[2]-v[2])^2
)

distance(
    (ones(3), zeros(3)),
    (zeros(3), ones(3))
)

0.0

In [99]:
function policy_eval(x, model; T=1000, τ_η=1e-8)
    n = length(x)
    V_U_0, V_E_0 = zeros(n), zeros(n)
    
    # local V_U_1, V_E_1
    for i in 1:T

        V_U_1, V_E_1 = value_update(V_U_0, V_E_0, x, model)
        # compute successive approximation errors
        η = distance(
            (V_U_1, V_E_1),
            (V_U_0, V_E_0)
        )
        if η<τ_η
            break # get out of the for loop
        end
        V_U_0, V_E_0 = V_U_1, V_E_1

    end

    return V_U_0, V_E_0
end

policy_eval (generic function with 1 method)

In [83]:
policy_eval(x_0, model)

1:0.41178149802215236
2:0.3454536656155482
3:0.2884728398813436
4:0.25279279833645685
5:0.22908532182379376
6:0.2122296137570257
7:0.19941994584039033
8:0.18909381645452292
9:0.18035843205753282
10:0.17268891571822756
11:0.1657677619310827
12:0.15939737425786862
13:0.15345114701197146
14:0.14784544537422492
15:0.14252322686260735
16:0.13744429544092157
17:0.13257940232588034
18:0.12790661000208317
19:0.12340900493145222
20:0.11907322450710894
21:0.11488848291853279
22:0.1108459082069814
23:0.10693807765833939
24:0.10315868287343095
25:0.09950228210261303
26:0.09596411313405545
27:0.09253994950919033
28:0.08922598864028099
29:0.08601876401376557
30:0.08291507595654818
31:0.07991193693865264
32:0.07700652839189819
33:0.07419616672344348
34:0.07147827670537131
35:0.0688503707937719
36:0.06631003321447991
37:0.06385490787380715
38:0.06148268932819292
39:0.05919111618798304
40:0.05697796644536251
41:0.05484105431038297
42:0.05277822821616802
43:0.05078736971795263
44:0.04886639306300609
45:

([-3.084359692900152, -3.3614962473857597, -3.3614962473857597], [-2.980433484968049, -1.7831548979472962, -0.70008467289815])

1.  **Define a function
    `bellman_step(V_E::Vector, V_U::Vector, p::Parameters)::Tuple{Vector, Vector, Vector}`
    which returns updated values, together with improved policy rules.**

In [49]:

(zeros(Bool, 3))

3-element Vector{Bool}:
 0
 0
 0

In [92]:
function bellman_step(V_U, V_E, model)

    #extract parameters form model
    (;β, λ, w, p, cbar) = model

    n = length(V_U) # here 3

    # initialize return variables
    n_V_E = V_E*0
    n_V_U = V_U*0
    x = zeros(Bool, n)

    # continuation value of being unemployed tomorrow
    # EVU = sum(V_U'*p)
    EVU = sum( (V_U[k]*p[k] for k=1:n) )

    # update employed values
    for k=1:n
        n_V_E[k] = log(w[k]) + (1-λ)*β*V_E[k] + λ*β*EVU
    end

    # update unemployed values
    for k=1:n
        #what if I accept:
        V_accept = log(cbar) + β*V_E[k]
        #what if I reject:
        V_reject = log(cbar) + β*EVU

        if V_accept>V_reject
            n_V_U[k] = V_accept
            x[k] = true
        else # if reject
            n_V_U[k] = V_reject
            x[k] = false
        end
    end

    return n_V_U, n_V_E, x
end

bellman_step (generic function with 1 method)

In [53]:
bellman_step(V_U_0, V_E_0, model)

([-0.0951435513142097, -0.0951435513142097, -0.031143551314209705], [-0.007760515657826278, 0.0976, 0.28411017980432496], Bool[0, 0, 1])

1.  **Implement Value Function Iteration**

In [93]:
function vfi(model; T=1000, τ_η=1e-8, verbose=false)
    
    n = length(model.w)
    V_U_0, V_E_0 = zeros(n), zeros(n)
    
    local x_1
    η_0 = NaN
    for i in 1:T

        V_U_1, V_E_1, x_1 = bellman_step(V_U_0, V_E_0, model)
        # compute successive approximation errors
        η = distance(
            (V_U_1, V_E_1),
            (V_U_0, V_E_0)
        )
        λ = η / η_0
        verbose ? (@show (i,η,λ)) : nothing
        if η<τ_η
            break # get out of the for loop
        end
        η_0 = η
        V_U_0, V_E_0 = V_U_1, V_E_1

    end

    return V_U_0, V_E_0, x_1
end

vfi (generic function with 1 method)

In [77]:
vfi(model; verbose=true)

(i, η, λ) = (1, 0.41178149802215236, NaN)
(i, η, λ) = (2, 0.18955616020190472, 0.46033190202175445)
(i, η, λ) = (3, 0.1743435805221963, 0.9197463186450664)
(i, η, λ) = (4, 0.1313743587844406, 0.7535371155676985)
(i, η, λ) = (5, 0.11739275606321094, 0.8935743409094715)
(i, η, λ) = (6, 0.1083925140301558, 0.9233322196796462)
(i, η, λ) = (7, 0.09974323800703834, 0.9202041201783441)
(i, η, λ) = (8, 0.09162892370385649, 0.9186479758897613)
(i, η, λ) = (9, 0.08418839942230115, 0.9187972096496188)
(i, η, λ) = (10, 0.07742317191816776, 0.9196418087223867)
(i, η, λ) = (11, 0.07128976550490142, 0.9207807396505399)
(i, η, λ) = (12, 0.06573496500400663, 0.922081375053578)
(i, η, λ) = (13, 0.06070622935222232, 0.9234998352630476)
(i, η, λ) = (14, 0.056627288512807315, 0.9328085291585371)
(i, η, λ) = (15, 0.05560223195275818, 0.9818981874822189)
(i, η, λ) = (16, 0.05466414931105856, 0.9831286873070734)
(i, η, λ) = (17, 0.053251412788989795, 0.9741560686505923)
(i, η, λ) = (18, 0.051501168919065976, 

([0.4158342892103646, 0.4158342892103646, 1.1651371859598578], [-0.8342230398289807, 0.363055547191771, 1.4461257722409178], Bool[0, 0, 1])

1.  **Implement Policy Iteration and compare rates of convergence.**

In [97]:
function policy_iteration(model; T=1000, τ_η=1e-8, verbose=false)
    
    n = length(model.w)
    V_U_0, V_E_0 = zeros(n), zeros(n)
    
    local x_1
    η_0 = NaN
    for i in 1:T

        V_U_, V_E_, x_1 = bellman_step(V_U_0, V_E_0, model)
        
        V_U_1, V_E_1 = policy_eval(x_1, model)

        # compute successive approximation errors
        η = distance(
            (V_U_1, V_E_1),
            (V_U_0, V_E_0)
        )
        λ = η / η_0
        verbose ? (@show (i,η,λ, x_1)) : nothing
        if η<τ_η
            break # get out of the for loop
        end
        η_0 = η
        V_U_0, V_E_0 = V_U_1, V_E_1

    end

    return V_U_0, V_E_0, x_1
end

policy_iteration (generic function with 1 method)

In [98]:
policy_iteration(model; verbose=true)

(i, η, λ, x_1) = (1, 11.155295764214697, NaN, Bool[0, 0, 0])
(i, η, λ, x_1) = (2, 10.050777979466549, 0.900987135787887, Bool[1, 1, 1])
(i, η, λ, x_1) = (3, 2.104636943993661, 0.20940040147074923, Bool[0, 1, 1])
(i, η, λ, x_1) = (4, 0.6128475729454325, 0.29118921184692376, Bool[0, 0, 1])
(i, η, λ, x_1) = (5, 0.0, 0.0, Bool[0, 0, 1])


([0.41583429078597567, 0.41583429078597567, 1.165137187535469], [-0.8342230382533695, 0.3630555487673824, 1.446125773816529], Bool[0, 0, 1])

1.  **Discuss the Effects of the Parameters**

In [105]:
policy_iteration(
    merge(
        model,
        (;λ=0.5)
    )
)[3]

3-element Vector{Bool}:
 0
 1
 1

In [110]:
policy_iteration(
    merge(
        model,
        (;cbar=0.2)
    )
)[3]

# decreasing unemployment benefits makes job-seekers accept lower paid jobs

3-element Vector{Bool}:
 1
 1
 1

In [114]:
policy_iteration(
    merge(
        model,
        (;β=0.9)
    )
)[3]
# more impatient: tend to accept any offer

3-element Vector{Bool}:
 0
 1
 1